Data sources: [2019](https://web.archive.org/web/20220811151927/https://www.cdc.gov/nchs/fastats/life-expectancy.htm), [2020](https://www.cdc.gov/nchs/fastats/life-expectancy.htm)<br>
[List of U.S. states and territories by life expectancy](https://en.wikipedia.org/wiki/List_of_U.S._states_and_territories_by_life_expectancy) <i>([alternative design](https://en.wikipedia.org/wiki/User:Lady3mlnm/List_of_U.S._states_and_territories_by_life_expectancy_(alternative)))</i> / [Продолжительность жизни в штатах США](https://ru.wikipedia.org/wiki/Продолжительность_жизни_в_штатах_США)<br>
[MapChart](https://www.mapchart.net/usa.html)<br>
Data for states are available at CDC since 2018.

In [2]:
import pandas as pd
import math
import re
from collections import namedtuple

import sys
sys.path.append("..")
import mal_moduls_private.mal_total as mal

In [3]:
# load data for single years from according files and concat them into united dataFrams
def load_single(year):
    df_current = pd.read_csv(f'data/USA_states_{year}.csv', sep='\t', index_col='Area', usecols=['Area', 'Total', 'Male', 'Female'])
    return df_current.rename(columns={'Total'  : f'{year}_t',
                                      'Male'   : f'{year}_m',
                                      'Female' : f'{year}_f'})

df = pd.concat([load_single(year) for year in [2018, 2019, 2020, 2021]], axis='columns')
df.index.name = ''
df.head(3)

,2018_t,2018_m,2018_f,2019_t,2019_m,2019_f,2020_t,2020_m,2020_f,2021_t,2021_m,2021_f
,,,,,,,,,,,,
Hawaii,81.0,78.0,84.0,80.9,78.0,83.9,80.7,77.6,83.8,79.9,77.0,83.1
California,80.8,78.4,83.1,80.9,78.4,83.3,79.0,76.2,82.0,78.3,75.3,81.4
New York,80.5,78.1,82.8,80.7,78.2,83.1,77.7,74.8,80.7,79.0,76.3,81.6


In [4]:
# sort values
df.sort_values(by=['2019_t', '2019_m', '2019_f'], ascending=False, inplace=True)
df = pd.concat([df.loc[['United States']], df.drop('United States')])

# calculate sex gap
df.insert(loc=3, column='2018_fΔm' ,  value=(df['2018_f']-df['2018_m']).round(1))
df.insert(loc=4, column='2018→2019',  value=(df['2019_t']-df['2018_t']).round(1))
df.insert(loc=8, column='2019_fΔm' ,  value=(df['2019_f']-df['2019_m']).round(1))
df.insert(loc=9, column='2019→2020',  value=(df['2020_t']-df['2019_t']).round(1))
df.insert(loc=13, column='2020_fΔm' ,  value=(df['2020_f']-df['2020_m']).round(1))
df.insert(loc=14, column='2020→2021',  value=(df['2021_t']-df['2020_t']).round(1))
df.insert(loc=18, column='2021_fΔm' ,  value=(df['2021_f']-df['2021_m']).round(1))
df.insert(loc=19, column='2019→2021',  value=(df['2021_t']-df['2019_t']).round(1))

df.rename(index={'United States': '– US –'}, inplace=True)

print(len(df))
df

52


,2018_t,2018_m,2018_f,2018_fΔm,2018→2019,2019_t,2019_m,2019_f,2019_fΔm,2019→2020,2020_t,2020_m,2020_f,2020_fΔm,2020→2021,2021_t,2021_m,2021_f,2021_fΔm,2019→2021
,,,,,,,,,,,,,,,,,,,,
– US –,78.7,76.2,81.2,5.0,0.1,78.8,76.3,81.4,5.1,-1.8,77.0,74.2,79.9,5.7,-0.6,76.4,73.5,79.3,5.8,-2.4
California,80.8,78.4,83.1,4.7,0.1,80.9,78.4,83.3,4.9,-1.9,79.0,76.2,82.0,5.8,-0.7,78.3,75.3,81.4,6.1,-2.6
Hawaii,81.0,78.0,84.0,6.0,-0.1,80.9,78.0,83.9,5.9,-0.2,80.7,77.6,83.8,6.2,-0.8,79.9,77.0,83.1,6.1,-1.0
New York,80.5,78.1,82.8,4.7,0.2,80.7,78.2,83.1,4.9,-3.0,77.7,74.8,80.7,5.9,1.3,79.0,76.3,81.6,5.3,-1.7
Minnesota,80.5,78.4,82.6,4.2,-0.1,80.4,78.3,82.6,4.3,-1.3,79.1,76.8,81.4,4.6,-0.3,78.8,76.3,81.4,5.1,-1.6
Massachusetts,80.1,77.7,82.5,4.8,0.3,80.4,77.9,82.8,4.9,-1.4,79.0,76.4,81.5,5.1,0.6,79.6,76.9,82.2,5.3,-0.8
Connecticut,80.4,77.9,82.9,5.0,-0.1,80.3,77.7,82.8,5.1,-1.9,78.4,75.6,81.3,5.7,0.8,79.2,76.3,82.0,5.7,-1.1
New Jersey,79.8,77.3,82.3,5.0,0.3,80.1,77.6,82.5,4.9,-2.6,77.5,74.6,80.5,5.9,1.5,79.0,76.3,81.6,5.3,-1.1
Washington,80.0,77.9,82.1,4.2,0.0,80.0,77.9,82.1,4.2,-0.8,79.2,76.9,81.6,4.7,-1.0,78.2,75.8,80.8,5.0,-1.8


<br>
<br>

In [6]:
mal.min_and_max_values(df[['2018_t', '2018→2019', '2019_t', '2019_m', '2019_f', '2019_fΔm', '2019→2020', '2020_t', '2020→2021', '2021_t', '2019→2021']],
                       row_center=['– US –'], nmb=5, max_lng=11)

Number of records: 52


,2018_t,2018→2019,2019_t,2019_m,2019_f,2019_fΔm,2019→2020,2020_t,2020→2021,2021_t,2019→2021
max,81.0 -Hawaii,0.5 -Vermont,80.9 -California,78.4 -California,83.9 -Hawaii,6.4 -Mississippi,-0.2 -Hawaii,80.7 -Hawaii,1.5 -New Jersey,79.9 -Hawaii,-0.8 -Massachuse…
max_2,80.8 -California,0.5 -Idaho,80.9 -Hawaii,78.3 -Minnesota,83.3 -California,6.2 -New Mexico,-0.4 -New Hampsh…,79.2 -Washington,1.3 -New York,79.6 -Massachuse…,-0.9 -New Hampsh…
max_3,80.5 -New York,0.3 -Massachuse…,80.7 -New York,78.2 -New York,83.1 -New York,6.0 -Alabama,-0.5 -Maine,79.1 -Minnesota,0.8 -Connecticut,79.2 -Connecticut,-1.0 -Hawaii
max_4,80.5 -Minnesota,0.3 -New Jersey,80.4 -Minnesota,78.0 -Hawaii,82.8 -Massachuse…,5.9 -Hawaii,-0.8 -Washington,79.0 -California,0.7 -North Dako…,79.0 -New York,-1.0 -Rhode Isla…
max_5,80.4 -Connecticut,0.3 -New Hampsh…,80.4 -Massachuse…,78.0 -Utah,82.8 -Connecticut,5.9 -Delaware,-0.8 -Oregon,79.0 -Massachuse…,0.6 -Massachuse…,79.0 -New Jersey,-1.1 -Connecticut
– US –,– 78.7 –,– 0.1 –,– 78.8 –,– 76.3 –,– 81.4 –,– 5.1 –,– -1.8 –,– 77.0 –,– -0.6 –,– 76.4 –,– -2.4 –
min_5,75.5 -Tennessee,-0.3 -Montana,75.6 -Tennessee,72.8 -Tennessee,78.3 -Oklahoma,4.2 -Nebraska,-2.5 -Arizona,73.5 -Kentucky,-1.4 -Florida,72.3 -Kentucky,-3.5 -Mississippi
min_4,75.3 -Kentucky,-0.3 -Rhode Isla…,75.5 -Kentucky,72.8 -Louisiana,78.2 -Alabama,4.2 -Washington,-2.6 -Louisiana,73.2 -Alabama,-1.4 -Oregon,72.2 -Louisiana,-3.5 -West Virgi…
min_3,75.1 -Alabama,-0.4 -Wyoming,75.2 -Alabama,72.2 -Alabama,78.0 -Kentucky,4.1 -Alaska,-2.6 -New Jersey,73.1 -Louisiana,-1.5 -New Mexico,72.0 -Alabama,-3.5 -Louisiana
min_2,74.6 -Mississippi,-0.5 -South Dako…,74.5 -West Virgi…,71.9 -West Virgi…,77.6 -Mississippi,4.0 -Idaho,-2.7 -District o…,72.8 -West Virgi…,-1.8 -West Virgi…,71.0 -West Virgi…,-3.8 -Arizona


<br>
<br>

In [8]:
# for state in sorted(df.index.to_list()):
#     print(f"    '{state}': {{'en': ('', ''), 'ru': ('', '')}},")

In [9]:
dd_replacement = {
    '– US –': {'en': ('US on average', ''), 'ru': ('США в среднем', '')},
    'Alabama': {'en': ('Alabama', 'Alabama'), 'ru': ('Алаба́ма', 'Алабама')},
    'Alaska': {'en': ('Alaska', 'Alaska'), 'ru': ('Аля́ска', 'Аляска')},
    'Arizona': {'en': ('Arizona', 'Arizona'), 'ru': ('Аризо́на', 'Аризона')},
    'Arkansas': {'en': ('Arkansas', 'Arkansas'), 'ru': ('Арканза́с (Арка́нзас)', 'Арканзас')},
    'California': {'en': ('California', 'California'), 'ru': ('Калифо́рния', 'Калифорния')},
    'Colorado': {'en': ('Colorado', 'Colorado'), 'ru': ('Колора́до', 'Колорадо')},
    'Connecticut': {'en': ('Connecticut', 'Connecticut'), 'ru': ('Конне́ктикут', 'Коннектикут')},
    'Delaware': {'en': ('Delaware', 'Delaware'), 'ru': ('Де́лавэр', 'Делавэр')},
    'District of Columbia': {'en': ('Washington, D.C.', 'Washington, D.C.'), 'ru': ('Вашингто́н (Ва́шингтон)', 'Вашингтон')},
    'Florida': {'en': ('Florida', 'Florida'), 'ru': ('Флори́да', 'Флорида')},
    'Georgia': {'en': ('Georgia', 'Georgia (U.S. state)'), 'ru': ('Джо́рджия', 'Джорджия')},
    'Hawaii': {'en': ('Hawaii', 'Hawaii'), 'ru': ('Гава́йи', 'Гавайи')},
    'Idaho': {'en': ('Idaho', 'Idaho'), 'ru': ('Айда́хо (А́йдахо)', 'Айдахо')},
    'Illinois': {'en': ('Illinois', 'Illinois'), 'ru': ('Иллино́йс', 'Иллинойс')},
    'Indiana': {'en': ('Indiana', 'Indiana'), 'ru': ('Индиа́на', 'Индиана')},
    'Iowa': {'en': ('Iowa', 'Iowa'), 'ru': ('А́йова (Айо́ва)', 'Айова')},
    'Kansas': {'en': ('Kansas', 'Kansas'), 'ru': ('Ка́нзас (Канза́с)', 'Канзас')},
    'Kentucky': {'en': ('Kentucky', 'Kentucky'), 'ru': ('Кенту́кки', 'Кентукки')},
    'Louisiana': {'en': ('Louisiana', 'Louisiana'), 'ru': ('Луизиа́на', 'Луизиана')},
    'Maine': {'en': ('Maine', 'Maine'), 'ru': ('Мэн', 'Мэн (штат)')},
    'Maryland': {'en': ('Maryland', 'Maryland'), 'ru': ('Мэ́риленд', 'Мэриленд')},
    'Massachusetts': {'en': ('Massachusetts', 'Massachusetts'), 'ru': ('Массачу́сетс', 'Массачусетс')},
    'Michigan': {'en': ('Michigan', 'Michigan'), 'ru': ('Мичига́н', 'Мичиган')},
    'Minnesota': {'en': ('Minnesota', 'Minnesota'), 'ru': ('Миннесо́та', 'Миннесота')},
    'Mississippi': {'en': ('Mississippi', 'Mississippi'), 'ru': ('Миссиси́пи', 'Миссисипи (штат)')},
    'Missouri': {'en': ('Missouri', 'Missouri'), 'ru': ('Миссу́ри', 'Миссури (штат)')},
    'Montana': {'en': ('Montana', 'Montana'), 'ru': ('Монта́на', 'Монтана')},
    'Nebraska': {'en': ('Nebraska', 'Nebraska'), 'ru': ('Небра́ска', 'Небраска')},
    'Nevada': {'en': ('Nevada', 'Nevada'), 'ru': ('Нева́да', 'Невада')},
    'New Hampshire': {'en': ('New Hampshire', 'New Hampshire'), 'ru': ('Нью-Гэ́мпшир', 'Нью-Гэмпшир')},
    'New Jersey': {'en': ('New Jersey', 'New Jersey'), 'ru': ('Нью-Дже́рси', 'Нью-Джерси')},
    'New Mexico': {'en': ('New Mexico', 'New Mexico'), 'ru': ('Нью-Ме́ксико', 'Нью-Мексико')},
    'New York': {'en': ('New York', 'New York (state)'), 'ru': ('Нью-Йо́рк (штат)', 'Нью-Йорк (штат)')},
    'North Carolina': {'en': ('North Carolina', 'North Carolina'), 'ru': ('Северная Кароли́на', 'Северная Каролина')},
    'North Dakota': {'en': ('North Dakota', 'North Dakota'), 'ru': ('Северная Дако́та', 'Северная Дакота')},
    'Ohio': {'en': ('Ohio', 'Ohio'), 'ru': ('Ога́йо', 'Огайо')},
    'Oklahoma': {'en': ('Oklahoma', 'Oklahoma'), 'ru': ('Оклахо́ма', 'Оклахома')},
    'Oregon': {'en': ('Oregon', 'Oregon'), 'ru': ('Орего́н', 'Орегон')},
    'Pennsylvania': {'en': ('Pennsylvania', 'Pennsylvania'), 'ru': ('Пенсильва́ния', 'Пенсильвания')},
    'Rhode Island': {'en': ('Rhode Island', 'Rhode Island'), 'ru': ('Род-А́йленд', 'Род-Айленд')},
    'South Carolina': {'en': ('South Carolina', 'South Carolina'), 'ru': ('Южная Кароли́на', 'Южная Каролина')},
    'South Dakota': {'en': ('South Dakota', 'South Dakota'), 'ru': ('Южная Дако́та', 'Южная Дакота')},
    'Tennessee': {'en': ('Tennessee', 'Tennessee'), 'ru': ('Теннесси́', 'Теннесси')},
    'Texas': {'en': ('Texas', 'Texas'), 'ru': ('Теха́с', 'Техас')},
    'Utah': {'en': ('Utah', 'Utah'), 'ru': ('Ю́та', 'Юта')},
    'Vermont': {'en': ('Vermont', 'Vermont'), 'ru': ('Вермо́нт', 'Вермонт')},
    'Virginia': {'en': ('Virginia', 'Virginia'), 'ru': ('Вирги́ния (Вирджи́ния)', 'Виргиния')},
    'Washington': {'en': ('Washington (state)', 'Washington (state)'), 'ru': ('Вашингто́н (штат)', 'Вашингтон (штат)')},
    'West Virginia': {'en': ('West Virginia', 'West Virginia'), 'ru': ('Западная Вирги́ния', 'Западная Виргиния')},
    'Wisconsin': {'en': ('Wisconsin', 'Wisconsin'), 'ru': ('Виско́нсин', 'Висконсин')},
    'Wyoming': {'en': ('Wyoming', 'Wyoming'), 'ru': ('Вайо́минг', 'Вайоминг')}
}

In [10]:
# create code for placing info in Wikipedia
def create_table_v1(df, file_header, lang='ru'):

    def if_value(x, prec=1):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)

    # def chval(x, prec=1, *, add_par=''):  # change_value
    #     return f'style="padding-right:4ex;{add_par}"|—' if math.isnan(x) else \
    #            f'style="padding-right:4ex;color:darkgreen;{add_par}"|{x:0.{prec}f}' if x>0 else \
    #            f'style="padding-right:4ex;color:crimson;{add_par}"|−{-x:0.{prec}f}' if x<0 else \
    #            f'style="padding-right:4ex;color:darkgray;{add_par}"|{x:0.{prec}f}'
    
    # def chval_bold(x, prec=1, *, add_par=''):  # change_value
    #     return '—' if math.isnan(x) else \
    #            f'style="padding-right:4ex;color:darkgreen;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
    #            f'style="padding-right:4ex;color:crimson;{add_par}"|\'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
    #            f'style="padding-right:4ex;color:darkgray;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\''
    
    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]    
        if ser.name == '– US –':
            st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2019_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2019_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2019_f"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["2019_fΔm"])}\'\'\' ' + \
                  f'||style="padding-right:2ex;border-left-width:2px;"| \'\'\'{if_value(ser["2019→2020"])}\'\'\' ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2020_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2020_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2020_f"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["2020_fΔm"])}\'\'\' ' + \
                  f'||style="padding-right:2ex;border-left-width:2px;"| \'\'\'{if_value(ser["2020→2021"])}\'\'\' ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2021_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2021_f"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["2021_fΔm"])}\'\'\' ' + \
                  f'||style="padding-right:2ex;border-left-width:2px;"| \'\'\'{if_value(ser["2019→2021"])}\'\'\''
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2019_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2019_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2019_f"])} ' + \
                  f'|| {if_value(ser["2019_fΔm"])} ' + \
                  f'||style="padding-right:2ex;border-left-width:2px;"| {if_value(ser["2019→2020"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2020_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2020_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2020_f"])} ' + \
                  f'|| {if_value(ser["2020_fΔm"])} ' + \
                  f'||style="padding-right:2ex;border-left-width:2px;"| {if_value(ser["2020→2021"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2021_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2021_f"])} ' + \
                  f'|| {if_value(ser["2021_fΔm"])} ' + \
                  f'||style="padding-right:2ex;border-left-width:2px;"| {if_value(ser["2019→2021"])}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        # st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —') \
           .replace(';"| \'\'\'—\'\'\'', ';color:silver;"| \'\'\'—\'\'\'')

    return st


table_code = create_table_v1(df, file_header='USA_header -2021, v1, ru.txt', lang='ru')
with open('output/Table code for USA states -v1 -ru.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

table_code = create_table_v1(df, file_header='USA_header -2021, v1, en.txt', lang='en')
# write the code to file
with open('output/Table code for USA states -v1 -en.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [11]:
# create code for placing info in Wikipedia
def create_table_v2(df, file_header, lang='ru'):

    def if_value(x, prec=1):
        return '—' if math.isnan(x) else \
               f"{x:0.{prec}f}"  if x>=0 else \
               f"−{-x:0.{prec}f}"                #"{x:0.{prec}f}".format(x, prec)

    # def chval(x, prec=1, *, add_par=''):  # change_value
    #     return f'style="padding-right:4ex;{add_par}"|—' if math.isnan(x) else \
    #            f'style="padding-right:4ex;color:darkgreen;{add_par}"|{x:0.{prec}f}' if x>0 else \
    #            f'style="padding-right:4ex;color:crimson;{add_par}"|−{-x:0.{prec}f}' if x<0 else \
    #            f'style="padding-right:4ex;color:darkgray;{add_par}"|{x:0.{prec}f}'
    
    # def chval_bold(x, prec=1, *, add_par=''):  # change_value
    #     return '—' if math.isnan(x) else \
    #            f'style="padding-right:4ex;color:darkgreen;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\'' if x>0 else \
    #            f'style="padding-right:4ex;color:crimson;{add_par}"|\'\'\'−{-x:0.{prec}f}\'\'\'' if x<0 else \
    #            f'style="padding-right:4ex;color:darkgray;{add_par}"|\'\'\'{x:0.{prec}f}\'\'\''
    
    with open('design/' + file_header, mode='r', encoding="utf-8") as fh:
        table_header = fh.read()

    st = ''
    for i in range(len(df)):
        ser = df.iloc[i]    
        if ser.name == '– US –':
            st += '\n' + '|-class=static-row-header\n' + \
                  f'| \'\'\'{dd_replacement[ser.name][lang][0]}\'\'\' ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2018_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2018_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2018_f"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["2018_fΔm"])}\'\'\' ' + \
                  f'||style="padding-right:2ex;border-left-width:2px;"| \'\'\'{if_value(ser["2018→2019"])}\'\'\' ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2019_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2019_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2019_f"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["2019_fΔm"])}\'\'\' ' + \
                  f'||style="padding-right:2ex;border-left-width:2px;"| \'\'\'{if_value(ser["2019→2021"])}\'\'\' ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| \'\'\'{if_value(ser["2021_m"])}\'\'\' ' + \
                  f'||style="background:#fee7f6;"| \'\'\'{if_value(ser["2021_f"])}\'\'\' ' + \
                  f'|| \'\'\'{if_value(ser["2021_fΔm"])}\'\'\''
        else:
            name_link = dd_replacement[ser.name][lang][1]
            name_visible = dd_replacement[ser.name][lang][0]
            name_inserted = name_link if name_link == name_visible else f"{name_link}|{name_visible}"
            st += '\n' + '|-\n' + \
                  f'| [[{name_inserted}]] ' + \
                  f'||style="background:#e0ffd8;"| \'\'\'{if_value(ser["2018_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2018_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2018_f"])} ' + \
                  f'|| {if_value(ser["2018_fΔm"])} ' + \
                  f'||style="padding-right:2ex;border-left-width:2px;"| {if_value(ser["2018→2019"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2019_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2019_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2019_f"])} ' + \
                  f'|| {if_value(ser["2019_fΔm"])} ' + \
                  f'||style="padding-right:2ex;border-left-width:2px;"| {if_value(ser["2019→2021"])} ' + \
                  f'||style="background:#e0ffd8;border-left-width:2px;"| \'\'\'{if_value(ser["2021_t"])}\'\'\' ' + \
                  f'||style="background:#eaf3ff;"| {if_value(ser["2021_m"])} ' + \
                  f'||style="background:#fee7f6;"| {if_value(ser["2021_f"])} ' + \
                  f'|| {if_value(ser["2021_fΔm"])}'

    if lang == 'ru':
        st = re.sub('(?<=\\d)\\.(?=\\d)', ',', st)  # replace . to comma, if this . is between two digits
        # st = st.replace('padding-right:1,5ex;', 'padding-right:1.5ex;')

    st = table_header + st + '\n|}'
    
    # gray color for missing values
    st = st.replace(';"| —', ';color:silver;"| —') \
           .replace(';"| \'\'\'—\'\'\'', ';color:silver;"| \'\'\'—\'\'\'')

    return st


table_code = create_table_v2(df, file_header='USA_header -2021, v2, ru.txt', lang='ru')
with open('output/Table code for USA states -v2 -ru.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

table_code = create_table_v2(df, file_header='USA_header -2021, v2, en.txt', lang='en')
# write the code to file
with open('output/Table code for USA states -v2 -en.txt', 'w', encoding="utf-8") as fh:
    fh.write(table_code)

In [12]:
# assert False

<br>
<br>
<br>
<hr>

<h3>Map creation</h3>

In [14]:
SELECTED_YEAR = 2019

In [15]:
CountryGroup = namedtuple('CountryGroup', ['group_label', 'color', 'countries'])

In [16]:
# for state in sorted(df.index.to_list()):
#     print(f"               '{state}' : ''")

In [17]:
df_map = df.copy()                                    \
           .drop(['– US –']) \
           .rename(index={
               'Alabama' : 'AL',
               'Alaska' : 'AK',
               'Arizona' : 'AZ',
               'Arkansas' : 'AR',
               'California' : 'CA',
               'Colorado' : 'CO',
               'Connecticut' : 'CT',
               'Delaware' : 'DE',
               'District of Columbia' : 'DC',
               'Florida' : 'FL',
               'Georgia' : 'GA',
               'Hawaii' : 'HI',
               'Idaho' : 'ID',
               'Illinois' : 'IL',
               'Indiana' : 'IN',
               'Iowa' : 'IA',
               'Kansas' : 'KS',
               'Kentucky' : 'KY',
               'Louisiana' : 'LA',
               'Maine' : 'ME',
               'Maryland' : 'MD',
               'Massachusetts' : 'MA',
               'Michigan' : 'MI',
               'Minnesota' : 'MN',
               'Mississippi' : 'MS',
               'Missouri' : 'MO',
               'Montana' : 'MT',
               'Nebraska' : 'NE',
               'Nevada' : 'NV',
               'New Hampshire' : 'NH',
               'New Jersey' : 'NJ',
               'New Mexico' : 'NM',
               'New York' : 'NY',
               'North Carolina' : 'NC',
               'North Dakota' : 'ND',
               'Ohio' : 'OH',
               'Oklahoma' : 'OK',
               'Oregon' : 'OR',
               'Pennsylvania' : 'PA',
               'Rhode Island' : 'RI',
               'South Carolina' : 'SC',
               'South Dakota' : 'SD',
               'Tennessee' : 'TN',
               'Texas' : 'TX',
               'Utah' : 'UT',
               'Vermont' : 'VT',
               'Virginia' : 'VA',
               'Washington' : 'WA',
               'West Virginia' : 'WV',
               'Wisconsin' : 'WI',
               'Wyoming' : 'WY'
           })

df_map.head(3).fillna('')

,2018_t,2018_m,2018_f,2018_fΔm,2018→2019,2019_t,2019_m,2019_f,2019_fΔm,2019→2020,2020_t,2020_m,2020_f,2020_fΔm,2020→2021,2021_t,2021_m,2021_f,2021_fΔm,2019→2021
,,,,,,,,,,,,,,,,,,,,
CA,80.8,78.4,83.1,4.7,0.1,80.9,78.4,83.3,4.9,-1.9,79.0,76.2,82.0,5.8,-0.7,78.3,75.3,81.4,6.1,-2.6
HI,81.0,78.0,84.0,6.0,-0.1,80.9,78.0,83.9,5.9,-0.2,80.7,77.6,83.8,6.2,-0.8,79.9,77.0,83.1,6.1,-1.0
NY,80.5,78.1,82.8,4.7,0.2,80.7,78.2,83.1,4.9,-3.0,77.7,74.8,80.7,5.9,1.3,79.0,76.3,81.6,5.3,-1.7


In [18]:
dd_legend = {
    # '80.5–81.0' : '002000',
    '80.5–80.9' : '002000',
    '80.0–80.4' : '004800',
    '79.5–79.9' : '006800',
    '79.0–79.4' : '009000',
    '78.5–78.9' : '00b800',
    '78.0–78.4' : '00e000',
    '77.5–77.9' : '00ff00',
    '77.0–77.4' : 'b8ff00',
    '76.5–76.9' : 'ffff00',
    '76.0–76.4' : 'ffe000',
    '75.5–75.9' : 'ffc000',
    '75.0–75.4' : 'ffa000',
    '74.5–74.9' : 'ff8000',
    '74.0–74.4' : 'ff5000',
    '73.5–73.9' : 'ff0000',
    '73.0–73.4' : 'd00000',
    '72.5–72.9' : 'a00000',
    '72.0–72.4' : '680000',
    '71.5–71.9' : '400000',
    '70.9–71.4' : '300000'
}

def create_legend_code(dd_legend):
    for k, v in dd_legend.items():
        print(f"{{{{Legend|#{v}|{k}}}}}")

create_legend_code(dd_legend)

{{Legend|#002000|80.5–80.9}}
{{Legend|#004800|80.0–80.4}}
{{Legend|#006800|79.5–79.9}}
{{Legend|#009000|79.0–79.4}}
{{Legend|#00b800|78.5–78.9}}
{{Legend|#00e000|78.0–78.4}}
{{Legend|#00ff00|77.5–77.9}}
{{Legend|#b8ff00|77.0–77.4}}
{{Legend|#ffff00|76.5–76.9}}
{{Legend|#ffe000|76.0–76.4}}
{{Legend|#ffc000|75.5–75.9}}
{{Legend|#ffa000|75.0–75.4}}
{{Legend|#ff8000|74.5–74.9}}
{{Legend|#ff5000|74.0–74.4}}
{{Legend|#ff0000|73.5–73.9}}
{{Legend|#d00000|73.0–73.4}}
{{Legend|#a00000|72.5–72.9}}
{{Legend|#680000|72.0–72.4}}
{{Legend|#400000|71.5–71.9}}
{{Legend|#300000|70.9–71.4}}


In [19]:
df_grouped = mal.bin_values_in_dataframe(df_map, f"{SELECTED_YEAR}_t", prec=1, step = 0.5)

df_grouped['group_label'] = df_grouped['group_label']
# df_grouped['group_label'] = df_grouped['group_label'].map(lambda st: '70.9–71.4' if st < '71.5–71.9' else st)
# df_grouped['group_label'] = df_grouped['group_label'].map(lambda st: '80.5–81.0' if st > '80.0–80.4' else st)

df_grouped

Range: 74.40 – 80.90   (MS – CA)
Number of groups: 13
Number of values: 51


,2019_t,group_label
,,
CA,80.9,80.5–80.9
HI,80.9,80.5–80.9
NY,80.7,80.5–80.9
MN,80.4,80.0–80.4
MA,80.4,80.0–80.4
CT,80.3,80.0–80.4
NJ,80.1,80.0–80.4
WA,80.0,80.0–80.4
CO,80.0,80.0–80.4


In [20]:
def extract_indexes(subdf, dd_legend = dd_legend):
    group_label = subdf['group_label'].iloc[0]
    countries = subdf.index.to_list()
    color = (dd_legend[group_label])
    
    ls_grouping.append(CountryGroup(group_label=group_label, countries=countries, color=color))

    return pd.Series([color, countries], index=['color', 'regions'])


ls_grouping = []
df_grouped = df_grouped.groupby(['group_label'])[['group_label']].apply(extract_indexes).loc[::-1]

df_grouped

,color,regions
group_label,,
80.5–80.9,002000,"[CA, HI, NY]"
80.0–80.4,004800,"[MN, MA, CT, NJ, WA, CO]"
79.5–79.9,006800,"[VT, UT, OR, RI, ID]"
79.0–79.4,009000,"[NH, WI, NE, VA, IA, IL, FL]"
78.5–78.9,00b800,"[ND, AZ, TX, MD]"
78.0–78.4,00e000,"[SD, MT, ME, PA, KS, DE, NV, MI, DC]"
77.5–77.9,00ff00,"[AK, WY, NC]"
77.0–77.4,b8ff00,"[GA, IN]"
76.5–76.9,ffff00,"[NM, OH, MO, SC]"


In [21]:
def create_map_code_regions(ls_grouping, title=''):
    st = '{"groups":{'
    for group_label, color, regions in ls_grouping[::-1]:
        st_ls_regions = '"' + '","'.join(regions) + '"'
        st_ls_regions = st_ls_regions.replace(' ', '_')
        st += f'"#{color}":{{"label":"{group_label}","paths":[{st_ls_regions}]}},'

    st = st[:-1] + '},"title":"' + title + \
         '","hidden":[],"background":"#fff","borders":"#000","legendFont":"Century Gothic","legendFontColor":"#000","legendBgColor":"#00000000","legendBoxShape":"square","legendBorderColor":"#00000000","legendWidth":249.00238095238115,"areBordersShown":true,"defaultColor":"#d1dbdd","labelsColor":"#000000","labelsFont":"Arial","strokeWidth":"medium","areLabelsShown":true,"uncoloredScriptColor":"#ffff33","v5":true,"usTerritoriesShown":false,"usFasShown":false,"splitStates":{},"legendPosition":"custom","legendX":1591.17619047619,"legendY":442.81428571428546,"canvasWidth":1861,"canvasHeight":1303,"legendSize":"custom","legendScale":1.25,"legendStatus":"show","scalingPatterns":true,"legendRowsSameColor":true,"legendColumnCount":1}'
    
    return st


st = create_map_code_regions(ls_grouping, title=str(SELECTED_YEAR))
print(st)

{"groups":{"#002000":{"label":"80.5–80.9","paths":["CA","HI","NY"]},"#004800":{"label":"80.0–80.4","paths":["MN","MA","CT","NJ","WA","CO"]},"#006800":{"label":"79.5–79.9","paths":["VT","UT","OR","RI","ID"]},"#009000":{"label":"79.0–79.4","paths":["NH","WI","NE","VA","IA","IL","FL"]},"#00b800":{"label":"78.5–78.9","paths":["ND","AZ","TX","MD"]},"#00e000":{"label":"78.0–78.4","paths":["SD","MT","ME","PA","KS","DE","NV","MI","DC"]},"#00ff00":{"label":"77.5–77.9","paths":["AK","WY","NC"]},"#b8ff00":{"label":"77.0–77.4","paths":["GA","IN"]},"#ffff00":{"label":"76.5–76.9","paths":["NM","OH","MO","SC"]},"#ffc000":{"label":"75.5–75.9","paths":["OK","AR","LA","TN","KY"]},"#ffa000":{"label":"75.0–75.4","paths":["AL"]},"#ff8000":{"label":"74.5–74.9","paths":["WV"]},"#ff5000":{"label":"74.0–74.4","paths":["MS"]}},"title":"2019","hidden":[],"background":"#fff","borders":"#000","legendFont":"Century Gothic","legendFontColor":"#000","legendBgColor":"#00000000","legendBoxShape":"square","legendBorderC